# Part 03 — Embeddings: How LLMs Represent Meaning

Embeddings are the bridge between raw text and neural network math.

```
"I love cats!"
      │  tokenize
 [2, 3, 4, 5]  ← arbitrary integer IDs (meaningless to a network)
      │  embedding lookup  (vocab_size × d_model matrix)
 [[-0.21,  0.44, -0.11, ...],   ← 768-dim vector for "I"
  [ 0.67, -0.23,  0.55, ...],   ← 768-dim vector for "love"
  [ 0.12,  0.89, -0.34, ...],   ← 768-dim vector for "cats"
  [-0.55,  0.11,  0.22, ...]]   ← 768-dim vector for "!"
      │  transformer layers
  context-aware representations
```

### What this notebook covers
| Section | Key idea |
|---------|----------|
| One-hot vs Dense | Why IDs and sparse vectors fail |
| Static embeddings | One fixed vector per word (Word2Vec, GloVe) |
| Contextual embeddings | Same word, different context → different vector (BERT) |
| Sentence embeddings | One vector per sentence for similarity / search |
| Visualizing embedding space | PCA to see semantic clusters |

In [ ]:
import torch
import torch.nn as nn

# ── Token IDs → Embedding Vectors ────────────────────────────────────────
# nn.Embedding is a learned lookup table: vocab_size × embedding_dim
# Each row = one token's learned vector

vocab_size    = 50000   # number of unique tokens in the vocabulary
embedding_dim = 768     # size of each token's vector (GPT-2 uses 768)

embedding_layer = nn.Embedding(vocab_size, embedding_dim)

# "I love cats!" → token_ids (simplified; real BPE tokenizer gives different IDs)
token_ids = [2, 3, 4, 5]

# Always wrap in an outer list to add the batch dimension
embeddings = embedding_layer(torch.tensor([token_ids]))

print(f"Input token IDs: {token_ids}")
print(f"Embedding shape: {embeddings.shape}")
# → torch.Size([1, 4, 768])
#                │  │  └── each token becomes a 768-dim vector
#                │  └───── 4 tokens in this sentence
#                └──────── batch size = 1

# Memory used by this embedding layer
params    = vocab_size * embedding_dim
memory_gb = (params * 4) / (1024**3)   # float32 = 4 bytes
print(f"\nEmbedding layer: {params:,} parameters  ({memory_gb:.3f} GB in float32)")
print(f"  = vocab_size({vocab_size:,}) × embedding_dim({embedding_dim})")

## The Batch Dimension — Why It Matters

Every model call expects shape `[batch_size, seq_len, d_model]`.  
Forgetting the outer `[]` is one of the most common bugs.

```python
# WRONG — model sees 4 separate single-token samples
token_ids = [1045, 2293, 8870, 999]
embedding_layer(torch.tensor(token_ids))
# → shape [4, 768]  (4 "batches" of 1 token each)

# CORRECT — model sees 1 sample with 4 tokens
embedding_layer(torch.tensor([token_ids]))
# → shape [1, 4, 768]  (1 batch, 4 tokens, 768 dims)
```

> In production, batch_size > 1 lets you process multiple sentences simultaneously on a GPU — much faster than one at a time.

---
## Static vs Contextual Embeddings

| Type | Example | Key property |
|------|---------|-------------|
| **Static** | Word2Vec, GloVe | One fixed vector per word, regardless of context |
| **Contextual** | BERT, GPT-2 | Vector changes based on surrounding words |

Static: `"bank"` → same vector in every sentence  
Contextual: `"bank"` → different vector in *"river bank"* vs *"bank deposit"*

### Method 1 — Static Word Embeddings (from a pretrained model)  
Directly index `model.embeddings.word_embeddings` — no context used.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

text = "I love cats!"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# STEP 1: TOKENIZATION
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")
# Output: ['i', 'love', 'cats', '!']

token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {token_ids}")
# Output: [1045, 2293, 8870, 999]

# STEP 2: EMBEDDING
model = AutoModel.from_pretrained("bert-base-uncased")

input_ids = torch.tensor([token_ids]) # Add batch dimension
with torch.no_grad(): # don't calculate gradient
    embeddings = model.embeddings.word_embeddings(input_ids)

print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding: {embeddings}")
# Output: 
# torch.Size([1, 4, 768]) --> [batch_size, sequence_length, embedding_dimension]
# Embedding(30522, 768, padding_idx=0)
##          |      |    |
##          |      |    └── Special padding token ID
##          |      └──── Embedding dimension (vector size)
##          └────────── Vocabulary size (total number of tokens)

### Method 2 — Contextual Embeddings (full forward pass through BERT)

The same token run through all 12 BERT layers produces a **context-aware** vector.  
`"bank"` in a finance sentence and `"bank"` in a geography sentence → different vectors.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Two sentences with same word "bank"
sentences = [
    "I went to the bank to deposit money",
    "I sat by the river bank"
]

for sentence in sentences:
    # Tokenize
    inputs = tokenizer(
        sentence, 
        return_tensors="pt", 
        padding=True,
        add_special_tokens=True
    )
    
    # Get contextual embeddings
    with torch.no_grad():
        outputs = model(**inputs)
        # WITHOUT .mean(dim=1) - TOKEN-LEVEL EMBEDDINGS  
        embeddings = outputs.last_hidden_state
        print(embeddings.shape)
    
    # Find "bank" token
    tokens = tokenizer.tokenize(sentence)
    bank_idx = tokens.index("bank")
    
    print(f"Sentence: {sentence}")
    print(f"'bank' embedding: {embeddings[0][bank_idx+1][:5]}...")  # +1 for [CLS]

# Output:
# torch.Size([1, 10, 768]) --> [batch_size, sequence_length, hidden_size] - ONE VECTOR PER TOKEN
# Sentence: I went to the bank to deposit money
# 'bank' embedding: tensor([ 0.4884, -0.2425,  0.0583, -0.1536,  1.0020])...

# torch.Size([1, 8, 768]) --> [batch_size, sequence_length, hidden_size] - ONE VECTOR PER TOKEN
# Sentence: I sat by the river bank
# 'bank' embedding: tensor([-0.3252, -0.7429, -0.3339, -0.0776, -0.3205])...

---
## Practical Use Cases

### 1 — Sentence Similarity (Mean Pooling)

`mean(last_hidden_state, dim=1)` collapses `[batch, seq_len, 768]` → `[batch, 768]`.  
This gives one vector per sentence you can compare with cosine similarity.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model     = AutoModel.from_pretrained("bert-base-uncased")

sentences = [
    "I love machine learning",
    "Machine learning is amazing",
    "I hate vegetables",
]

sentence_embeddings = []
for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt", padding=True,
                       truncation=True, add_special_tokens=True)
    with torch.no_grad():
        outputs = model(**inputs)
        # Mean pooling: average all token vectors → one vector per sentence
        emb = outputs.last_hidden_state.mean(dim=1)   # [1, 768]
    sentence_embeddings.append(emb)

# Shape check
print(f"Per-sentence embedding shape: {sentence_embeddings[0].shape}")
# → torch.Size([1, 768])  — one 768-dim vector per sentence

print("\nCosine similarities:")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = F.cosine_similarity(sentence_embeddings[i], sentence_embeddings[j]).item()
        print(f"  {sim:.3f}  '{sentences[i][:25]}' vs '{sentences[j][:25]}'")

# Expected:
#   ~0.82  "I love machine learning" vs "Machine learning is amazing"  (topically similar)
#   ~0.79  "I love machine learning" vs "I hate vegetables"            (surface overlap "I ___ ...")
#   ~0.60  "Machine learning is amazing" vs "I hate vegetables"        (least similar)

#### 2. Using Embeddings for Classification

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn

class TextClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token embedding
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding)

# Initialize classifier
model = TextClassifier("bert-base-uncased", num_classes=2)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example usage
text = "This movie is great!"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
logits = model(inputs['input_ids'], inputs['attention_mask'])
print(f"Classification scores: {logits}")

#### 3. Extract Embeddings for Custom Tasks

In [ ]:
from transformers import AutoTokenizer, AutoModel
import numpy as np

def extract_embeddings(texts, model_name="bert-base-uncased"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    all_embeddings = []
    
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = model(**inputs)
            # Mean pooling across tokens
            embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
            all_embeddings.append(embedding)
    
    return np.array(all_embeddings)

# Example
texts = [
    "Natural language processing",
    "Computer vision",
    "Machine learning algorithms"
]

embeddings = extract_embeddings(texts)
print(f"Embeddings shape: {embeddings.shape}")  # (3, 768)

# Now you can use these for clustering, similarity search, etc.
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2)
clusters = kmeans.fit_predict(embeddings)
print(f"Cluster assignments: {clusters}")

---
## Model Comparison — Embedding Dimensions & Vocab Sizes

### Which model to use?

| Model | Best for | Vocab | d_model | Total params |
|-------|----------|-------|---------|--------------|
| `bert-base-uncased` | General NLP, classification | 30,522 | 768 | 110M |
| `bert-large-uncased` | Higher accuracy (slower) | 30,522 | 1,024 | 340M |
| `roberta-base` | Stronger BERT (better pretraining) | 50,265 | 768 | 125M |
| `distilbert-base-uncased` | Fast/lightweight (~60% size of BERT) | 30,522 | 768 | 66M |
| `sentence-transformers/all-MiniLM-L6-v2` | Sentence similarity / search | 30,522 | 384 | 22M |
| `microsoft/codebert-base` | Code + natural language | 50,265 | 768 | 125M |
| `gpt2` | Text generation (decoder-only) | 50,257 | 768 | 124M |
| `meta-llama/Llama-2-7b` | Generalist LLM | 32,000 | 4,096 | 7B |

> **Rule of thumb:** Use `sentence-transformers/all-MiniLM-L6-v2` for similarity/search tasks — it's specifically trained for that and 5× smaller than BERT.

---
## Seeing Embeddings Work — Similarity Demo

BERT's embedding layer (before contextual processing) already clusters  
similar words together, because it was pretrained on billions of sentences.

The raw token IDs give **meaningless distances** — "cat" (4937) is arithmetically  
closer to "car" (2482) than to "dog" (3899). Embeddings fix this.


## Real Example: Why Embeddings Work

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# ── Visualize embedding space via PCA ────────────────────────────────────
# Use BERT's static word embeddings (before contextual layers) to see clusters

from transformers import AutoModel, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model     = AutoModel.from_pretrained("bert-base-uncased")
emb_layer = model.embeddings.word_embeddings

word_groups = {
    "Animals":    ["cat", "dog", "lion", "tiger", "wolf", "bear", "fox", "rabbit"],
    "Vehicles":   ["car", "truck", "bus", "motorcycle", "bicycle", "van", "lorry", "jeep"],
    "Royalty":    ["king", "queen", "prince", "princess", "duke", "earl", "knight", "lord"],
    "Countries":  ["france", "germany", "japan", "china", "india", "brazil", "canada", "spain"],
}

group_colors = {"Animals": "#e74c3c", "Vehicles": "#3498db",
                "Royalty": "#f39c12", "Countries": "#2ecc71"}

all_words, all_groups = [], []
for group, words in word_groups.items():
    all_words.extend(words)
    all_groups.extend([group] * len(words))

# Get embeddings
ids = tokenizer.convert_tokens_to_ids(all_words)
with torch.no_grad():
    vecs = emb_layer(torch.tensor(ids)).numpy()   # [N, 768]

# Reduce to 2D with PCA
pca  = PCA(n_components=2, random_state=42)
xy   = pca.fit_transform(vecs)

# ── Plot ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_title("BERT Embedding Space — PCA to 2D\nEach cluster = semantically related words",
             fontsize=12, fontweight='bold')

for group, color in group_colors.items():
    mask = [i for i, g in enumerate(all_groups) if g == group]
    ax.scatter(xy[mask, 0], xy[mask, 1], c=color, s=120, label=group,
               edgecolors='white', linewidths=0.8, zorder=3)
    for i in mask:
        ax.annotate(all_words[i], (xy[i, 0], xy[i, 1]),
                    textcoords="offset points", xytext=(5, 4),
                    fontsize=7.5, color=color, alpha=0.9)

ax.legend(fontsize=10, loc='upper right')
ax.set_xlabel(f"PC1  ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=9)
ax.set_ylabel(f"PC2  ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=9)
ax.grid(True, alpha=0.2)
ax.axhline(0, color='gray', alpha=0.3, linewidth=0.5)
ax.axvline(0, color='gray', alpha=0.3, linewidth=0.5)

plt.tight_layout()
plt.savefig("images/embedding_pca.png", dpi=120, bbox_inches='tight')
plt.show()
print("Observation: Words of the same type cluster together in embedding space,")
print("even though the model was never explicitly told which words are 'animals'.")

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Get token IDs
cat_id = tokenizer.convert_tokens_to_ids(['cat'])[0]
dog_id = tokenizer.convert_tokens_to_ids(['dog'])[0] 
car_id = tokenizer.convert_tokens_to_ids(['car'])[0]

print(f"cat ID: {cat_id}")  # 4937
print(f"dog ID: {dog_id}")  # 3899
print(f"car ID: {car_id}")  # 2482

# Method 1: Using raw IDs (WRONG WAY)
print("\n=== USING RAW TOKEN IDs ===")
def similarity_with_ids(id1, id2):
    return 1 / (1 + abs(id1 - id2))  # Inverse distance

cat_dog_sim_ids = similarity_with_ids(cat_id, dog_id)
cat_car_sim_ids = similarity_with_ids(cat_id, car_id)

print(f"Cat-Dog similarity (IDs): {cat_dog_sim_ids:.4f}")
print(f"Cat-Car similarity (IDs): {cat_car_sim_ids:.4f}")

# Method 2: Using embeddings (CORRECT WAY)
print("\n=== USING EMBEDDINGS ===")
from transformers import AutoModel

model = AutoModel.from_pretrained('bert-base-uncased')
embedding_layer = model.embeddings.word_embeddings

# Get embeddings
cat_emb = embedding_layer(torch.tensor([cat_id]))
dog_emb = embedding_layer(torch.tensor([dog_id]))
car_emb = embedding_layer(torch.tensor([car_id]))

# Calculate cosine similarity
def cosine_similarity(a, b):
    return torch.cosine_similarity(a, b, dim=0)

cat_dog_sim_emb = cosine_similarity(cat_emb.squeeze(), dog_emb.squeeze())
cat_car_sim_emb = cosine_similarity(cat_emb.squeeze(), car_emb.squeeze())

print(f"Cat-Dog similarity (embeddings): {cat_dog_sim_emb:.4f}")
print(f"Cat-Car similarity (embeddings): {cat_car_sim_emb:.4f}")

---
## Why Not Just Use Token IDs? — The Representation Problem

Token IDs are arbitrary integers. They carry **no semantic information**:

```
cat      → 4937
dog      → 3899
car      → 2482
```

The gap between `cat(4937)` and `dog(3899)` = 1038. Between `cat(4937)` and `car(2482)` = 2455.  
By raw distance, "cat" is _closer to car_ than to dog. That's meaningless.

### The evolution of word representations

| Method | Vector for "cat" | Dimensions | Captures meaning? |
|--------|-----------------|------------|-------------------|
| Raw ID | `4937` | 1 | No |
| One-hot | `[0,0,...,1,...,0]` | vocab_size (10k+) | No — all pairs equally distant |
| Word2Vec / GloVe | `[0.45, -0.12, 0.87, ...]` | 50–300 | Yes — trained on co-occurrence |
| BERT / GPT-2 | `[0.21, -0.44, ...]` (context-aware) | 768–4096 | Yes + context-aware |

**The core insight:** meaning is multi-dimensional.  
A single number can't encode: *is it an animal? is it domestic? how big? can it be a verb?*  
A 768-dim vector can encode all of these (implicitly, via training).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Visualize: One-hot vs Dense Embeddings ────────────────────────────────

VOCAB = ["cat", "dog", "car", "ship", "kitten", "puppy", "truck", "boat"]
VOCAB_SIZE = len(VOCAB)
DENSE_DIM  = 4   # 4D for visualization (real = 768)

np.random.seed(42)

# One-hot: sparse, no similarity information
one_hot = np.eye(VOCAB_SIZE)

# Dense: hand-crafted to show semantic clustering
#         [animal, domestic, land_vehicle, water_vehicle]
dense = np.array([
    [ 0.9,  0.8, -0.3, -0.3],   # cat
    [ 0.9,  0.8, -0.3, -0.3],   # dog    ← close to cat
    [-0.3, -0.3,  0.9,  0.1],   # car
    [-0.3, -0.2,  0.1,  0.9],   # ship
    [ 0.8,  0.7, -0.2, -0.2],   # kitten ← close to cat/dog
    [ 0.8,  0.7, -0.2, -0.2],   # puppy  ← close to cat/dog
    [-0.2, -0.3,  0.8,  0.1],   # truck  ← close to car
    [-0.2, -0.2,  0.1,  0.8],   # boat   ← close to ship
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("One-hot vs Dense Embeddings", fontsize=14, fontweight='bold')

# ── Left: One-hot heatmap ─────────────────────────────────────────────────
ax1 = axes[0]
im1 = ax1.imshow(one_hot, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax1.set_xticks(range(VOCAB_SIZE)); ax1.set_yticks(range(VOCAB_SIZE))
ax1.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=9)
ax1.set_yticklabels(VOCAB, fontsize=9)
ax1.set_title(f"One-hot  [{VOCAB_SIZE}×{VOCAB_SIZE}]\n"
              "All pairs are equally distant — dot(cat,dog)=0, dot(cat,car)=0", fontsize=9)
ax1.set_xlabel(f"Dimensions (vocab_size = {VOCAB_SIZE})")
plt.colorbar(im1, ax=ax1, fraction=0.046)

# ── Right: Dense embedding heatmap ───────────────────────────────────────
ax2 = axes[1]
im2 = ax2.imshow(dense, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
ax2.set_xticks(range(DENSE_DIM))
ax2.set_xticklabels(['animal', 'domestic', 'land_veh', 'water_veh'], fontsize=9)
ax2.set_yticks(range(VOCAB_SIZE)); ax2.set_yticklabels(VOCAB, fontsize=9)
ax2.set_title(f"Dense  [{VOCAB_SIZE}×{DENSE_DIM}]\n"
              "cat/dog/kitten/puppy cluster; car/truck cluster; ship/boat cluster", fontsize=9)
ax2.set_xlabel(f"Dimensions (d_model = {DENSE_DIM}, real models use 768+)")
plt.colorbar(im2, ax=ax2, fraction=0.046)
for i in range(VOCAB_SIZE):
    for j in range(DENSE_DIM):
        ax2.text(j, i, f"{dense[i,j]:.1f}", ha='center', va='center', fontsize=7,
                 color='black' if abs(dense[i,j]) < 0.6 else 'white')

plt.tight_layout()
plt.savefig("images/onehot_vs_dense.png", dpi=120, bbox_inches='tight')
plt.show()
print("Key observation: in the dense matrix, cat/dog/kitten/puppy have SIMILAR rows.")
print("In one-hot, every row is completely different from every other row.")

# ── Print cosine similarity comparison ───────────────────────────────────
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

pairs = [("cat","dog"), ("cat","kitten"), ("cat","car"), ("dog","puppy")]
print(f"\n{'Pair':<20} {'One-hot cos':>14} {'Dense cos':>12}  {'Relationship'}")
print("-" * 65)
for w1, w2 in pairs:
    i1, i2 = VOCAB.index(w1), VOCAB.index(w2)
    oh_sim = cosine_sim(one_hot[i1], one_hot[i2])
    de_sim = cosine_sim(dense[i1], dense[i2])
    rel = "similar" if de_sim > 0.8 else "unrelated"
    print(f"  {w1} vs {w2:<12} {oh_sim:>14.3f} {de_sim:>12.3f}  {rel}")

---
## Summary

### The three types of embeddings you'll use

| Type | How to get it | Shape | Use when |
|------|--------------|-------|----------|
| **Token embedding** | `model.embeddings.word_embeddings(ids)` | `[T, d]` | Inspecting vocabulary, probing |
| **Contextual token** | `outputs.last_hidden_state` | `[B, T, d]` | NER, QA, token classification |
| **Sentence embedding** | `.last_hidden_state.mean(dim=1)` or `[:, 0, :]` | `[B, d]` | Similarity, clustering, retrieval |

### Key numbers to remember

```
BERT-base:   vocab=30,522   d_model=768   embedding_layer=23M params
GPT-2 Small: vocab=50,257   d_model=768   embedding_layer=39M params
LLaMA-2-7B:  vocab=32,000   d_model=4,096 embedding_layer=131M params

Memory formula: vocab_size × d_model × 4 bytes  (float32)
```

### What makes embeddings powerful
1. **Geometry = semantics** — cosine distance between vectors reflects meaning distance
2. **Linearity** — `king − man + woman ≈ queen` works in embedding space
3. **Transferable** — pretrain once on massive data, fine-tune for your task
4. **Composable** — sentence vectors are meaningful averages of token vectors

> The embedding matrix is shared between the input lookup and the output unembedding (weight tying). This halves embedding parameters and enforces a consistent representation.